This script extends the TAME_DataSelection by additionally balancing pain score distributions. While the original selection ensured a balanced representation of gender and race/ethnicity at the participant level, this script further selects audio files from the chosen participants such that pain scores are approximately equally represented.

From the initial set of 1,941 available audio files (from the 20 selected participants based on gender/race), a subset of 1,000 files was selected. A balancing strategy was applied for the painscore. However, a perfectly balanced distribution is not feasible, as certain pain levels (e.g., pain score 10) occur only a limited number of times (n = 5). Enforcing strict balancing would therefore reduce the dataset to only 55 audio files (5 per pain level across 11 classes). Therefore I choose to select minimal 1000 files

In [1]:
# Standard libraries
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns
import shutil
import os

from scipy.io import wavfile
from scipy.signal import wiener

# Load data
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")
AUDIO_PATH = DATA_PATH / "mic1_trim_v1~"
META_AUDIO_PATH = DATA_PATH / "meta_audio.csv"
META_PARTICIPANT_PATH = DATA_PATH / "meta_participant.csv"
AUDIO_BASE_PATH = DATA_PATH / "mic1_trim_v1~" / "mic1_trim_v2"

# Check everything exists
assert DATA_PATH.exists(), "Data map niet gevonden!"
assert AUDIO_PATH.exists(), "Audio map niet gevonden!"
assert META_AUDIO_PATH.exists(), "meta_audio.csv niet gevonden!"
assert META_PARTICIPANT_PATH.exists(), "meta_participant.csv niet gevonden!"

print("The paths are correct")

# Load metadata
meta_audio = pd.read_csv(META_AUDIO_PATH)
meta_participant = pd.read_csv(META_PARTICIPANT_PATH)

print("\nMeta audio shape:", meta_audio.shape)
print("Meta participant shape:", meta_participant.shape)

# Collect audio files
audio_files = list(AUDIO_PATH.rglob("*.wav"))

print(f"\nNumber of audio files: {len(audio_files)}")


def load_meta_participant(file_path):
    """
    Load meta_participant
    """
    df = pd.read_csv(file_path)

    # Split columns with ','
    if df.shape[1] == 1:
        first_col = df.columns[0]
        split_df = df[first_col].str.split(",", expand=True)
        split_df.columns = first_col.split(",")
        df = split_df.copy()

    # Delete spaces
    df.columns = [col.strip() for col in df.columns]

    return df


def stratified_participant_sample(
    file_path,
    n=20,
    gender_col="GENDER",
    race_col="RACE/ETHNICITY",
    id_col="PID",
    random_state=42
):
    """
    Selects n participants while preserving the distribution of gender and race/ethnicity
    as closely as possible.
    """
    df = load_meta_participant(file_path).copy()

    needed_cols = [id_col, gender_col, race_col]
    for col in needed_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available columns: {list(df.columns)}")

    df = df.dropna(subset=[gender_col, race_col]).copy()

    if n > len(df):
        raise ValueError(f"You ask {n} participants, but there are {len(df)} available.")

    # strata gender and race
    df["stratum"] = df[gender_col].astype(str).str.strip() + " | " + df[race_col].astype(str).str.strip()

    # size per stratum
    stratum_counts = df["stratum"].value_counts().sort_index()
    proportions = stratum_counts / len(df)

    # initial allocation
    target_counts = proportions * n
    allocated = np.floor(target_counts).astype(int)

    # distribute remaining samples
    remainder = n - allocated.sum()
    fractional_parts = (target_counts - allocated).sort_values(ascending=False)

    for stratum in fractional_parts.index[:remainder]:
        allocated[stratum] += 1

    # ensure we never sample more than available
    allocated = pd.Series({
        stratum: min(allocated[stratum], stratum_counts[stratum])
        for stratum in allocated.index
    })

    # fill up if necessary due to rounding
    current_total = allocated.sum()
    if current_total < n:
        shortfall = n - current_total

        spare_capacity = pd.Series({
            stratum: stratum_counts[stratum] - allocated[stratum]
            for stratum in allocated.index
        }).sort_values(ascending=False)

        for stratum in spare_capacity.index:
            if shortfall == 0:
                break
            if spare_capacity[stratum] > 0:
                extra = min(spare_capacity[stratum], shortfall)
                allocated[stratum] += extra
                shortfall -= extra

    # draw the sample
    sampled_parts = []
    for stratum, k in allocated.items():
        if k > 0:
            subset = df[df["stratum"] == stratum]
            sampled_parts.append(subset.sample(n=k, random_state=random_state))

    sample_df = pd.concat(sampled_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    sample_df = sample_df.drop(columns=["stratum"])

    return sample_df


def print_sample_summary(sample_df, gender_col="GENDER", race_col="RACE/ETHNICITY", id_col="PID"):
    """
    Summary selected sample
    """
    print(f"Number of selected participants: {len(sample_df)}\n")

    print("Distribution gender:")
    print(sample_df[gender_col].value_counts(dropna=False))
    print()

    print("Distribution race/ethnicity:")
    print(sample_df[race_col].value_counts(dropna=False))
    print()

    print("Selection of participants:")
    print(sample_df[id_col].tolist())
    print()


# output
sample_20 = stratified_participant_sample(
    file_path=META_PARTICIPANT_PATH,
    n=20,
    random_state=42
)

print_sample_summary(sample_20)

# save
sample_20.to_csv("selected_participants_20.csv", index=False)


def load_meta_audio(file_path):
    """
    Load meta_audio.csv, and divide the columns if needed.
    """
    df = pd.read_csv(file_path)

    if df.shape[1] == 1:
        first_col = df.columns[0]
        split_df = df[first_col].str.split(",", expand=True)
        split_df.columns = [col.strip() for col in first_col.split(",")]
        df = split_df.copy()

    df.columns = [col.strip() for col in df.columns]
    return df


def select_valid_audio_rows(meta_audio_path, selected_participants_df):
    """
    Select rows from meta_audio for the chosen participants only,
    where NOTES is empty and ACTION LABEL equals 0.
    """
    meta_audio_df = load_meta_audio(meta_audio_path).copy()

    # selected PID list
    selected_pids = selected_participants_df["PID"].astype(str).tolist()

    # keep only chosen participants
    filtered_df = meta_audio_df[meta_audio_df["PID"].astype(str).isin(selected_pids)].copy()

    # make notes empty
    filtered_df["NOTES"] = filtered_df["NOTES"].fillna("").astype(str).str.strip()

    # ACTION label
    filtered_df["ACTION LABEL"] = pd.to_numeric(filtered_df["ACTION LABEL"], errors="coerce")

    # filter: NOTES empty and ACTION LABEL == 0
    filtered_df = filtered_df[
        (filtered_df["NOTES"] == "") &
        (filtered_df["ACTION LABEL"] == 0)
    ].copy()

    return filtered_df


def balance_audio_by_pain_score_soft(
    df,
    pain_col="PAIN LEVEL",
    target_total=1000, #number of files we want in the end
    max_ratio=1.5, #how many times a large class vs small class 
    random_state=42
):
    """
    Soft balancing of pain scores:
    - Try to reach target_total number of samples
    - Allow imbalance but limit dominance of large classes

    Parameters
    df : pd.DataFrame
    pain_col : str
    target_total : int
        Desired total number of samples
    max_ratio : float
        Maximum allowed ratio between largest and smallest class
    """

    data = df.copy()

    data[pain_col] = pd.to_numeric(data[pain_col], errors="coerce")
    data = data.dropna(subset=[pain_col]).copy()
    data[pain_col] = data[pain_col].astype(int)

    counts = data[pain_col].value_counts().sort_index()

    print("\nOriginal pain distribution:")
    print(counts)

    n_classes = len(counts)
    ideal_per_class = target_total // n_classes
    max_per_class = int(ideal_per_class * max_ratio) #maximum per class
    sampled_parts = []

    for pain_level, group in data.groupby(pain_col):

        available = len(group)

        # kies aantal samples
        n_samples = min(available, max_per_class)

        sampled = group.sample(n=n_samples, random_state=random_state)
        sampled_parts.append(sampled)

    balanced_df = pd.concat(sampled_parts)

    #Fill with random data if there isn't 1000 samples yet
    if len(balanced_df) < target_total:
        remaining = data.drop(balanced_df.index)
        extra_needed = target_total - len(balanced_df)

        if len(remaining) > 0:
            extra_samples = remaining.sample(
                n=min(extra_needed, len(remaining)),
                random_state=random_state
            )
            balanced_df = pd.concat([balanced_df, extra_samples])

    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nFinal number of samples:", len(balanced_df))

    print("\nNew pain distribution:")
    print(balanced_df[pain_col].value_counts().sort_index())

    return balanced_df


def build_audio_paths(filtered_audio_df, audio_base_path, extension=".wav"):
    """
    Build the expected audio file path for each selected row.
    Filename format:
    PID.COND.UTTNUM.UTTID.wav
    Stored in:
    audio_base_path / PID / filename
    """
    df = filtered_audio_df.copy()

    # make sure components are strings without spaces
    df["PID"] = df["PID"].astype(str).str.strip()
    df["COND"] = df["COND"].astype(str).str.strip()
    df["UTTNUM"] = df["UTTNUM"].astype(str).str.strip()
    df["UTTID"] = df["UTTID"].astype(str).str.strip()

    # create filename
    df["filename"] = (
        df["PID"] + "." +
        df["COND"] + "." +
        df["UTTNUM"] + "." +
        df["UTTID"] + extension
    )

    # full path
    df["file_path"] = df.apply(
        lambda row: os.path.join(audio_base_path, row["PID"], row["filename"]),
        axis=1
    )

    # check existence
    df["file_exists"] = df["file_path"].apply(os.path.exists)

    return df


# Step 1: all valid audio rows for selected participants
filtered_audio = select_valid_audio_rows(
    meta_audio_path=META_AUDIO_PATH,
    selected_participants_df=sample_20
)

# Step 2: balance on pain score
balanced_audio = balance_audio_by_pain_score_soft(
    df=filtered_audio,
    pain_col="PAIN LEVEL",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
)

# Step 3: build paths
audio_selection = build_audio_paths(
    filtered_audio_df=balanced_audio,
    audio_base_path=AUDIO_BASE_PATH,
    extension=".wav"
)

# Save as CSV
audio_selection_existing = audio_selection[audio_selection["file_exists"]].copy()
audio_selection_existing.to_csv("selected_audio_files_pain.csv", index=False)
print("Saved: selected_audio_files_pain.csv")

# Folder to save the selected original (non-Wiener) audio files
ORIGINAL_SELECTED_OUTPUT_DIR = os.path.join(DATA_PATH, "selected_original_audio_pain")
os.makedirs(ORIGINAL_SELECTED_OUTPUT_DIR, exist_ok=True)

print("\nOriginal selected audio output directory:")
print(ORIGINAL_SELECTED_OUTPUT_DIR)


def make_original_output_path(input_path, output_root):
    """
    Copy original selected audio to:
    output_root / PID / original_filename.wav
    """
    participant_id = os.path.basename(os.path.dirname(input_path))
    filename = os.path.basename(input_path)

    participant_output_dir = os.path.join(output_root, participant_id)
    os.makedirs(participant_output_dir, exist_ok=True)

    return os.path.join(participant_output_dir, filename)


# Copy selected original audio files to a separate folder
original_selected_rows = []

for _, row in audio_selection_existing.iterrows():
    input_path = row["file_path"]

    try:
        output_path = make_original_output_path(input_path, ORIGINAL_SELECTED_OUTPUT_DIR)

        # Copy original wav file
        shutil.copy2(input_path, output_path)

        row_dict = row.to_dict()
        row_dict["selected_original_file_path"] = output_path
        original_selected_rows.append(row_dict)

    except Exception as e:
        print(f"Error copying {input_path}: {e}")

# create dataframe and save as CSV
original_selected_audio_df = pd.DataFrame(original_selected_rows)
print("Number of copied original selected audio files:", len(original_selected_audio_df))
print(original_selected_audio_df.head())

original_selected_audio_df.to_csv("selected_audio_files_original_pain.csv", index=False)
print("Saved: selected_audio_files_original_pain.csv")


# Folder to save Wiener-filtered audio
WIENER_OUTPUT_DIR = os.path.join(DATA_PATH, "wiener_filtered_audio_pain")
os.makedirs(WIENER_OUTPUT_DIR, exist_ok=True)

print("\nWiener output directory:")
print(WIENER_OUTPUT_DIR)


def load_wav_file(file_path):
    """
    Load a WAV file and return sample rate and signal.
    """
    sample_rate, signal = wavfile.read(file_path)
    return sample_rate, signal


def apply_wiener_filter(signal, mysize=29, noise=None):
    """
    Apply Wiener filtering to an audio signal.
    """
    signal = signal.astype(np.float32)
    filtered_signal = wiener(signal, mysize=mysize, noise=noise)
    return filtered_signal


def save_wav_file(file_path, sample_rate, signal):
    """
    Save a WAV file as int16.
    """
    signal = np.clip(signal, -32768, 32767)
    signal = signal.astype(np.int16)
    wavfile.write(file_path, sample_rate, signal)


def make_output_path(input_path, output_root):
    """
    Save filtered audio in:
    output_root / PID / original_filename.wav
    """
    participant_id = os.path.basename(os.path.dirname(input_path))
    filename = os.path.basename(input_path)

    participant_output_dir = os.path.join(output_root, participant_id)
    os.makedirs(participant_output_dir, exist_ok=True)

    return os.path.join(participant_output_dir, filename)


processed_rows = []

for _, row in audio_selection_existing.iterrows():
    input_path = row["file_path"]

    try:
        sample_rate, signal = load_wav_file(input_path)

        # Apply Wiener filtering
        filtered_signal = apply_wiener_filter(signal, mysize=29)

        # Create output path
        output_path = make_output_path(input_path, WIENER_OUTPUT_DIR)

        # Save filtered file
        save_wav_file(output_path, sample_rate, filtered_signal)

        # Store row info
        row_dict = row.to_dict()
        row_dict["wiener_file_path"] = output_path
        processed_rows.append(row_dict)

    except Exception as e:
        print(f"Error processing {input_path}: {e}")

wiener_audio_df = pd.DataFrame(processed_rows)
print("Number of Wiener audio files:", len(wiener_audio_df))
wiener_audio_df.to_csv("selected_audio_files_wiener_pain.csv", index=False)
print("Saved: selected_audio_files_wiener_pain.csv")

The paths are correct

Meta audio shape: (7044, 9)
Meta participant shape: (51, 11)

Number of audio files: 14088
Number of selected participants: 20

Distribution gender:
GENDER
Woman         10
Man            8
Male           1
Non-binary     1
Name: count, dtype: int64

Distribution race/ethnicity:
RACE/ETHNICITY
Asian                11
White                 6
Hispanic/Latino       2
Two or more races     1
Name: count, dtype: int64

Selection of participants:
['p80330', 'p28030', 'p60145', 'p64560', 'p79665', 'p68870', 'p68340', 'p77560', 'p59520', 'p79550', 'p91315', 'p15965', 'p68625', 'p18785', 'p97630', 'p10085', 'p20960', 'p62650', 'p37540', 'p94215']


Original pain distribution:
PAIN LEVEL
0       5
1     908
2     107
3     148
4     273
5     122
6     144
7     138
8      71
9      20
10      5
Name: count, dtype: int64

Final number of samples: 1005

New pain distribution:
PAIN LEVEL
0       5
1     135
2     107
3     135
4     135
5     122
6     135
7     135
8      7

c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: divide by zero encountered in divide
  res *= (1 - noise / lVar)
c:\Users\marti\anaconda3\Lib\site-packages\scipy\signal\_signaltools.py:1659: RuntimeWarning: invalid value encountered in multiply
  res *= (1 - noise / lVar)


Number of Wiener audio files: 1005
Saved: selected_audio_files_wiener_pain.csv


Showing the pain levels 

In [3]:
def print_pain_distribution(df, pain_col="PAIN LEVEL", title="Pain level distribution"):
    """
    Print the distribution of pain levels.
    """
    if pain_col not in df.columns:
        raise ValueError(
            f"Pain score column '{pain_col}' not found. Available columns: {list(df.columns)}"
        )

    temp_df = df.copy()
    temp_df[pain_col] = pd.to_numeric(temp_df[pain_col], errors="coerce")
    temp_df = temp_df.dropna(subset=[pain_col]).copy()
    temp_df[pain_col] = temp_df[pain_col].astype(int)

    counts = temp_df[pain_col].value_counts().sort_index()
    percentages = (counts / counts.sum() * 100).round(2)

    summary_df = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    print(f"\n--- {title} ---")
    print(summary_df)

print_pain_distribution(
    filtered_audio,
    pain_col="PAIN LEVEL", 
    title="Pain level distribution after selecting 20 participants"
)

print("\nNumber of audio rows after pain balancing:", len(balanced_audio))

print_pain_distribution(
    balanced_audio,
    pain_col="PAIN LEVEL",   
    title="Pain level distribution after pain balancing"
)


--- Pain level distribution after selecting 20 participants ---
            count  percentage
PAIN LEVEL                   
0               5        0.26
1             908       46.78
2             107        5.51
3             148        7.62
4             273       14.06
5             122        6.29
6             144        7.42
7             138        7.11
8              71        3.66
9              20        1.03
10              5        0.26

Number of audio rows after pain balancing: 1005

--- Pain level distribution after pain balancing ---
            count  percentage
PAIN LEVEL                   
0               5        0.50
1             135       13.43
2             107       10.65
3             135       13.43
4             135       13.43
5             122       12.14
6             135       13.43
7             135       13.43
8              71        7.06
9              20        1.99
10              5        0.50
